# Line Follower ResNet18 XY Training + Bin Convert Workflow

这个 notebook 会把多个候选数据目录合并后用于训练和后续转换。
默认会按顺序检查这些目录：`/home/ubuntu/数据`、`/home/ubuntu/image_data`、`/home/ubuntu/jupyter_notebooks/data`、`/home/ubuntu/jupyter_notebooks/image_data`。
不存在的目录会自动跳过，重名图片会自动去重。

当前数据集的标签来自文件名，例如：`xy_025_474_xxx.jpg`。
这里把 `x,y` 当作两个连续值做回归，而不是分类。


In [1]:
from pathlib import Path
import json
import re
import shlex
import shutil
import subprocess
from statistics import mean, pstdev

NOTEBOOK_ROOT = Path('/home/ubuntu/jupyter_notebooks')
WORKDIR = Path('/home/ubuntu/10_model_convert')
DATA_DIRS = [
    Path('/home/ubuntu/数据'),
    Path('/home/ubuntu/image_data'),
    NOTEBOOK_ROOT / 'data',
    NOTEBOOK_ROOT / 'image_data',
]
WORK_IMAGE_DATASET_DIR = WORKDIR / 'mapper' / 'image_dataset'
MERGED_TRAIN_DATA_DIR = WORKDIR / 'merged_training_data'
TRAIN_SCRIPT = WORKDIR / 'train_resnet_xy.py'
EXPORT_SCRIPT = WORKDIR / 'export_resnet_xy_to_onnx.py'
PTH_FILE = Path('/home/ubuntu/best_line_follower_model_xy.pth')
STATS_FILE = WORKDIR / 'label_stats_xy.json'
SUMMARY_FILE = WORKDIR / 'training_summary_xy.json'
SPLIT_FILE = WORKDIR / 'train_split_xy.json'
OE_ROOT = Path('/home/ubuntu/horizon_x5_open_explorer_v1.2.8-py310_20240926')
SAMPLE_ROOT = OE_ROOT / 'samples/ai_toolchain/horizon_model_convert_sample/03_classification'
WORK_NAME = '10_model_convert'
MAPPER_DIR = SAMPLE_ROOT / WORK_NAME / 'mapper'
ONNX_FILE = 'best_line_follower_model_xy.onnx'
CONFIG_FILE = 'resnet18_config.yaml'
DOCKER_IMAGE = 'openexplorer/ai_toolchain_ubuntu_20_x5_cpu:v1.2.8-py310'
MARCH = 'bayes-e'

IMAGE_SIZE = 224
EPOCHS = 60
BATCH_SIZE = 32
WORKERS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.2
SEED = 42
USE_PRETRAINED = True
DEVICE = 'cuda'


def collect_dataset_files(pattern='*.jpg'):
    existing_dirs = []
    files = []
    seen_names = set()
    duplicate_names = []
    for directory in DATA_DIRS:
        if not directory.exists():
            continue
        existing_dirs.append(directory)
        for path in sorted(directory.glob(pattern)):
            if path.name in seen_names:
                duplicate_names.append(path.name)
                continue
            seen_names.add(path.name)
            files.append(path)
    if not existing_dirs:
        raise AssertionError(f'No dataset directory found. Checked: {DATA_DIRS}')
    if not files:
        raise AssertionError(f'No {pattern} files found in dataset dirs: {existing_dirs}')
    return existing_dirs, files, sorted(set(duplicate_names))


def rebuild_dataset_dir(target_dir: Path, files):
    target_dir.mkdir(parents=True, exist_ok=True)
    for path in target_dir.glob('*.jpg'):
        path.unlink()
    for src in files:
        shutil.copy2(src, target_dir / src.name)
    return len(list(target_dir.glob('*.jpg')))


print('WORKDIR =', WORKDIR)
print('DATA_DIRS =')
for d in DATA_DIRS:
    print(' -', d, '(exists)' if d.exists() else '(missing)')
print('MERGED_TRAIN_DATA_DIR =', MERGED_TRAIN_DATA_DIR)
print('WORK_IMAGE_DATASET_DIR =', WORK_IMAGE_DATASET_DIR)


def run(cmd: str):
    print(f'\n$ {cmd}\n')
    completed = subprocess.run(cmd, shell=True, executable='/bin/bash', text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')
    return completed


WORKDIR = /home/ubuntu/10_model_convert
DATA_DIRS =
 - /home/ubuntu/数据 (exists)
 - /home/ubuntu/image_data (missing)
 - /home/ubuntu/jupyter_notebooks/data (missing)
 - /home/ubuntu/jupyter_notebooks/image_data (missing)
MERGED_TRAIN_DATA_DIR = /home/ubuntu/10_model_convert/merged_training_data
WORK_IMAGE_DATASET_DIR = /home/ubuntu/10_model_convert/mapper/image_dataset


## 1. 环境和数据集检查

这里先确认：
- 工作目录、训练脚本、导出脚本存在
- 候选数据目录里至少有一个存在
- 合并后的图片文件名都符合 `xy_x_y_*.jpg` 规则
- 会打印实际参与训练的数据目录和总图片数


In [2]:
assert WORKDIR.exists(), f'WORKDIR not found: {WORKDIR}'
assert TRAIN_SCRIPT.exists(), f'TRAIN_SCRIPT not found: {TRAIN_SCRIPT}'
assert EXPORT_SCRIPT.exists(), f'EXPORT_SCRIPT not found: {EXPORT_SCRIPT}'

run('source ~/miniconda3/etc/profile.d/conda.sh && conda run -n yolo11 python -c "import torch, torchvision, onnx; print(torch.__version__); print(torch.cuda.is_available()); print(torchvision.__version__); print(onnx.__version__)"')

pattern = re.compile(r'^xy_(\d+)_(\d+)_')
existing_dirs, files, duplicate_names = collect_dataset_files('*.jpg')
xs = []
ys = []
for path in files:
    match = pattern.match(path.name)
    assert match, f'Unexpected filename: {path.name}'
    xs.append(int(match.group(1)))
    ys.append(int(match.group(2)))

print('dataset dirs =')
for d in existing_dirs:
    print(' -', d)
print('image_count =', len(files))
if duplicate_names:
    print('ignored duplicate filenames =', len(duplicate_names))
    for name in duplicate_names[:10]:
        print(' -', name)
print('x range     =', min(xs), '->', max(xs), '| mean =', round(mean(xs), 2), '| std =', round(pstdev(xs), 2))
print('y range     =', min(ys), '->', max(ys), '| mean =', round(mean(ys), 2), '| std =', round(pstdev(ys), 2))
print('sample files =')
for path in files[:5]:
    print(' -', path)



$ source ~/miniconda3/etc/profile.d/conda.sh && conda run -n yolo11 python -c "import torch, torchvision, onnx; print(torch.__version__); print(torch.cuda.is_available()); print(torchvision.__version__); print(onnx.__version__)"

2.6.0+cu124
True
0.21.0+cu124
1.21.0

dataset dirs =
 - /home/ubuntu/数据
image_count = 3875
x range     = 5 -> 1635 | mean = 694.41 | std = 546.45
y range     = 116 -> 750 | mean = 424.56 | std = 151.46
sample files =
 - /home/ubuntu/数据/xy_005_532_3838e7aa-32f8-11f0-b6bc-c0353254ea4e.jpg
 - /home/ubuntu/数据/xy_006_451_f637913b-19f3-11f1-94e9-acf23cd8a5be.jpg
 - /home/ubuntu/数据/xy_007_370_b59272b8-19f3-11f1-b4d8-acf23cd8a5be.jpg
 - /home/ubuntu/数据/xy_007_696_24f0f544-3af5-11f1-8113-00d49e7b40e3.jpg
 - /home/ubuntu/数据/xy_007_720_a4cc4ed4-3af5-11f1-a491-00d49e7b40e3.jpg


## 2. 启动训练

训练前会先把所有可用数据目录中的图片合并到 `10_model_convert/merged_training_data`，
再把这份合并后的目录传给训练脚本。

训练脚本会做这些事情：
- 从文件名解析 `x,y`
- 按训练/验证集切分
- 用 ResNet18 做 XY 回归训练
- 保存最佳权重、标签统计、训练摘要和切分记录


In [3]:
pretrained_flag = '--pretrained' if USE_PRETRAINED else ''
existing_dirs, files, duplicate_names = collect_dataset_files('*.jpg')
merged_count = rebuild_dataset_dir(MERGED_TRAIN_DATA_DIR, files)

print('training dataset dirs =')
for d in existing_dirs:
    print(' -', d)
print('merged training images =', merged_count)
if duplicate_names:
    print('ignored duplicate filenames =', len(duplicate_names))

cmd = (
    'source ~/miniconda3/etc/profile.d/conda.sh && '
    f'conda run -n yolo11 python {shlex.quote(str(TRAIN_SCRIPT))} '
    f'--dataset-dir {shlex.quote(str(MERGED_TRAIN_DATA_DIR))} '
    f'--output-pth {shlex.quote(str(PTH_FILE))} '
    f'--stats-file {shlex.quote(str(STATS_FILE))} '
    f'--summary-file {shlex.quote(str(SUMMARY_FILE))} '
    f'--split-file {shlex.quote(str(SPLIT_FILE))} '
    f'--epochs {EPOCHS} --batch-size {BATCH_SIZE} --workers {WORKERS} '
    f'--lr {LR} --weight-decay {WEIGHT_DECAY} --val-ratio {VAL_RATIO} '
    f'--image-size {IMAGE_SIZE} --seed {SEED} --device {DEVICE} {pretrained_flag}'
)
run(cmd)


training dataset dirs =
 - /home/ubuntu/数据
merged training images = 3875

$ source ~/miniconda3/etc/profile.d/conda.sh && conda run -n yolo11 python /home/ubuntu/10_model_convert/train_resnet_xy.py --dataset-dir /home/ubuntu/10_model_convert/merged_training_data --output-pth /home/ubuntu/best_line_follower_model_xy.pth --stats-file /home/ubuntu/10_model_convert/label_stats_xy.json --summary-file /home/ubuntu/10_model_convert/training_summary_xy.json --split-file /home/ubuntu/10_model_convert/train_split_xy.json --epochs 60 --batch-size 32 --workers 8 --lr 0.001 --weight-decay 0.0001 --val-ratio 0.2 --image-size 224 --seed 42 --device cuda --pretrained

epoch 001/60 | train_loss=0.1309 | val_loss=0.0455 | val_mae=(123.33, 35.41) | lr=0.000999
epoch 002/60 | train_loss=0.0514 | val_loss=0.0319 | val_mae=(72.34, 35.17) | lr=0.000997
epoch 003/60 | train_loss=0.0326 | val_loss=0.0485 | val_mae=(61.93, 49.94) | lr=0.000994
epoch 004/60 | train_loss=0.0328 | val_loss=0.0272 | val_mae=(54.10,

CompletedProcess(args='source ~/miniconda3/etc/profile.d/conda.sh && conda run -n yolo11 python /home/ubuntu/10_model_convert/train_resnet_xy.py --dataset-dir /home/ubuntu/10_model_convert/merged_training_data --output-pth /home/ubuntu/best_line_follower_model_xy.pth --stats-file /home/ubuntu/10_model_convert/label_stats_xy.json --summary-file /home/ubuntu/10_model_convert/training_summary_xy.json --split-file /home/ubuntu/10_model_convert/train_split_xy.json --epochs 60 --batch-size 32 --workers 8 --lr 0.001 --weight-decay 0.0001 --val-ratio 0.2 --image-size 224 --seed 42 --device cuda --pretrained', returncode=0, stdout='epoch 001/60 | train_loss=0.1309 | val_loss=0.0455 | val_mae=(123.33, 35.41) | lr=0.000999\nepoch 002/60 | train_loss=0.0514 | val_loss=0.0319 | val_mae=(72.34, 35.17) | lr=0.000997\nepoch 003/60 | train_loss=0.0326 | val_loss=0.0485 | val_mae=(61.93, 49.94) | lr=0.000994\nepoch 004/60 | train_loss=0.0328 | val_loss=0.0272 | val_mae=(54.10, 31.45) | lr=0.000989\nepoc

## 3. 查看训练产物


In [4]:
assert PTH_FILE.exists(), f'Checkpoint not found: {PTH_FILE}'
assert STATS_FILE.exists(), f'Stats file not found: {STATS_FILE}'
assert SUMMARY_FILE.exists(), f'Summary file not found: {SUMMARY_FILE}'
assert SPLIT_FILE.exists(), f'Split file not found: {SPLIT_FILE}'

stats = json.loads(STATS_FILE.read_text(encoding='utf-8'))
summary = json.loads(SUMMARY_FILE.read_text(encoding='utf-8'))

print('checkpoint =', PTH_FILE)
print('stats      =', STATS_FILE)
print('summary    =', SUMMARY_FILE)
print('best epoch =', summary['best_epoch'])
print('best loss  =', round(summary['best_val_loss'], 6))
if summary.get('history'):
    last = summary['history'][-1]
    print('last epoch =', last['epoch'])
    print('last val MAE =', round(last['val_mae_x'], 2), round(last['val_mae_y'], 2))
print('label mean =', stats['label_mean'])
print('label std  =', stats['label_std'])


checkpoint = /home/ubuntu/best_line_follower_model_xy.pth
stats      = /home/ubuntu/10_model_convert/label_stats_xy.json
summary    = /home/ubuntu/10_model_convert/training_summary_xy.json
best epoch = 44
best loss  = 0.012366
last epoch = 60
last val MAE = 35.55 18.59
label mean = [694.8787096774194, 425.5025806451613]
label std  = [547.1466903605726, 151.2986375856564]


## 4. 导出 ONNX

导出脚本会读取训练得到的 `label_stats_xy.json`，
把模型输出从“标准化坐标”恢复成“原始坐标”，再写成 ONNX。


In [5]:
cmd = (
    'source ~/miniconda3/etc/profile.d/conda.sh && '
    f'conda run -n yolo11 python {shlex.quote(str(EXPORT_SCRIPT))} '
    f'--pth {shlex.quote(str(PTH_FILE))} '
    f'--stats-file {shlex.quote(str(STATS_FILE))} '
    f'--onnx-out {shlex.quote(str(WORKDIR / "mapper" / ONNX_FILE))} '
    f'--image-size {IMAGE_SIZE}'
)
run(cmd)
assert (WORKDIR / 'mapper' / ONNX_FILE).exists(), f'ONNX not found: {WORKDIR / "mapper" / ONNX_FILE}'



$ source ~/miniconda3/etc/profile.d/conda.sh && conda run -n yolo11 python /home/ubuntu/10_model_convert/export_resnet_xy_to_onnx.py --pth /home/ubuntu/best_line_follower_model_xy.pth --stats-file /home/ubuntu/10_model_convert/label_stats_xy.json --onnx-out /home/ubuntu/10_model_convert/mapper/best_line_follower_model_xy.onnx --image-size 224

preview output = [586.4326782226562, 603.1336059570312]
onnx exported to /home/ubuntu/10_model_convert/mapper/best_line_follower_model_xy.onnx



## 5. 同步数据并复制到 Open Explorer sample 目录

这里会把实际参与训练的多目录数据重新合并，
同步到 `10_model_convert/mapper/image_dataset`，
然后继续复用服务器上已有的 Horizon 转换流程。


In [6]:
assert OE_ROOT.exists(), f'OE_ROOT not found: {OE_ROOT}'
assert SAMPLE_ROOT.exists(), f'SAMPLE_ROOT not found: {SAMPLE_ROOT}'

existing_dirs, source_files, duplicate_names = collect_dataset_files('*.jpg')
synced_count = rebuild_dataset_dir(WORK_IMAGE_DATASET_DIR, source_files)

print('source dataset dirs =')
for d in existing_dirs:
    print(' -', d)
print('synced image count =', synced_count)
if duplicate_names:
    print('ignored duplicate filenames =', len(duplicate_names))

dst_work = SAMPLE_ROOT / WORK_NAME
run(f'sudo rm -rf {shlex.quote(str(dst_work))}')
run(f'cp -a {shlex.quote(str(WORKDIR))} {shlex.quote(str(SAMPLE_ROOT))}')
run(f'sudo chown -R ubuntu:ubuntu {shlex.quote(str(dst_work))}')

assert MAPPER_DIR.exists(), f'MAPPER_DIR not found: {MAPPER_DIR}'
assert (MAPPER_DIR / CONFIG_FILE).exists(), f'Config not found: {MAPPER_DIR / CONFIG_FILE}'
assert (MAPPER_DIR / ONNX_FILE).exists(), f'ONNX not found: {MAPPER_DIR / ONNX_FILE}'
assert (MAPPER_DIR / 'image_dataset').exists(), f'image_dataset not found: {MAPPER_DIR / "image_dataset"}'


source dataset dirs =
 - /home/ubuntu/数据
synced image count = 3875

$ sudo rm -rf /home/ubuntu/horizon_x5_open_explorer_v1.2.8-py310_20240926/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert


$ cp -a /home/ubuntu/10_model_convert /home/ubuntu/horizon_x5_open_explorer_v1.2.8-py310_20240926/samples/ai_toolchain/horizon_model_convert_sample/03_classification


$ sudo chown -R ubuntu:ubuntu /home/ubuntu/horizon_x5_open_explorer_v1.2.8-py310_20240926/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert



## 6. 生成校准数据 + checker


In [7]:
cmd = f"sudo docker run --rm -v {OE_ROOT}:/open_explorer {DOCKER_IMAGE} bash -lc 'cd /open_explorer/samples/ai_toolchain/horizon_model_convert_sample/03_classification/{WORK_NAME}/mapper && rm -rf calibration_data_bgr_f32 && bash 02_preprocess.sh'"
run(cmd)
print('calibration files =', len(list((MAPPER_DIR / 'calibration_data_bgr_f32').glob('*.rgb'))))

cmd = f"sudo docker run --rm -v {OE_ROOT}:/open_explorer {DOCKER_IMAGE} bash -lc 'cd /open_explorer/samples/ai_toolchain/horizon_model_convert_sample/03_classification/{WORK_NAME}/mapper && hb_mapper checker --model-type onnx --model ./{ONNX_FILE} --march {MARCH}'"
run(cmd)



$ sudo docker run --rm -v /home/ubuntu/horizon_x5_open_explorer_v1.2.8-py310_20240926:/open_explorer openexplorer/ai_toolchain_ubuntu_20_x5_cpu:v1.2.8-py310 bash -lc 'cd /open_explorer/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert/mapper && rm -rf calibration_data_bgr_f32 && bash 02_preprocess.sh'

Warning please note that the data type is now determined by the name of the folder suffix
Warning if you need to set it explicitly, please configure the value of saved_data_type in the preprocess shell script
Regular preprocess
Init 10 processes
write:./calibration_data_bgr_f32/xy_007_696_24f0f544-3af5-11f1-8113-00d49e7b40e3.rgb
write:./calibration_data_bgr_f32/xy_018_629_964036fd-4068-11f1-a2c8-00d49e7b40e3.rgb
write:./calibration_data_bgr_f32/xy_023_695_25e5fe51-3af5-11f1-be82-00d49e7b40e3.rgb
write:./calibration_data_bgr_f32/xy_029_529_90ec9ee9-32f6-11f0-a46c-f43bd8ce20fb.rgb
write:./calibration_data_bgr_f32/xy_033_680_89f97176-4068-11f1-9070-00d49e

CompletedProcess(args="sudo docker run --rm -v /home/ubuntu/horizon_x5_open_explorer_v1.2.8-py310_20240926:/open_explorer openexplorer/ai_toolchain_ubuntu_20_x5_cpu:v1.2.8-py310 bash -lc 'cd /open_explorer/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert/mapper && hb_mapper checker --model-type onnx --model ./best_line_follower_model_xy.onnx --march bayes-e'", returncode=0, stdout='', stderr="2026-04-25 15:31:49,699 \x1bINFO\x1b log will be stored in /open_explorer/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert/mapper/hb_mapper_checker.log\n2026-04-25 15:31:49,699 \x1bINFO\x1b Start hb_mapper....\n2026-04-25 15:31:49,699 \x1bINFO\x1b hbdk version 3.49.15\n2026-04-25 15:31:49,699 \x1bINFO\x1b horizon_nn version 1.1.0\n2026-04-25 15:31:49,699 \x1bINFO\x1b hb_mapper version 1.24.3\n2026-04-25 15:31:49,748 \x1bINFO\x1b Model type: onnx\n2026-04-25 15:31:49,748 \x1bINFO\x1b input names []\n2026-04-25 15:31:49,748 \x1b

## 7. 生成 bin


In [8]:
cmd = f"sudo docker run --rm -v {OE_ROOT}:/open_explorer {DOCKER_IMAGE} bash -lc 'cd /open_explorer/samples/ai_toolchain/horizon_model_convert_sample/03_classification/{WORK_NAME}/mapper && bash 03_build.sh'"
run(cmd)
run(f"sudo chown -R ubuntu:ubuntu {shlex.quote(str(SAMPLE_ROOT / WORK_NAME))}")



$ sudo docker run --rm -v /home/ubuntu/horizon_x5_open_explorer_v1.2.8-py310_20240926:/open_explorer openexplorer/ai_toolchain_ubuntu_20_x5_cpu:v1.2.8-py310 bash -lc 'cd /open_explorer/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert/mapper && bash 03_build.sh'

c63bf477bfe29f50215df9e663c72434  /open_explorer/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert/mapper/calibration_data_bgr_f32/xy_005_532_3838e7aa-32f8-11f0-b6bc-c0353254ea4e.rgb

2026-04-25 15:31:59,213 INFO log will be stored in /open_explorer/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert/mapper/hb_mapper_makertbin.log
2026-04-25 15:31:59,214 INFO Start hb_mapper....
2026-04-25 15:31:59,214 INFO hbdk version 3.49.15
2026-04-25 15:31:59,214 INFO horizon_nn version 1.1.0
2026-04-25 15:31:59,214 INFO hb_mapper version 1.24.3
2026-04-25 15:31:59,214 INFO Start Model Convert....
2026-04-25 15:31:59,225 INFO Using onnx

CompletedProcess(args='sudo chown -R ubuntu:ubuntu /home/ubuntu/horizon_x5_open_explorer_v1.2.8-py310_20240926/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert', returncode=0, stdout='', stderr='')

## 8. 查看转换产物


In [9]:
from datetime import datetime
import shutil

output_dir = MAPPER_DIR / 'model_output'
assert output_dir.exists(), f'model_output not found: {output_dir}'
for path in sorted(output_dir.iterdir()):
    print(' -', path.name, f'({path.stat().st_size / 1024 / 1024:.2f} MB)')

bin_candidates = sorted(output_dir.glob('*.bin'))
assert bin_candidates, f'No .bin file found in: {output_dir}'
bin_file = bin_candidates[0]

easy_bin = Path('/home/ubuntu/model_latest.bin')
shutil.copy2(bin_file, easy_bin)

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
stamped = Path(f'/home/ubuntu/{bin_file.stem}_{stamp}.bin')
shutil.copy2(bin_file, stamped)

print('copied from:', bin_file)
print('copied to  :', easy_bin)
print('copied to  :', stamped)


 - main_graph_subgraph_0.html (2.15 MB)
 - main_graph_subgraph_0.json (0.00 MB)
 - resnet18_224x224_nv12.bin (10.38 MB)
 - resnet18_224x224_nv12_calibrated_model.onnx (42.70 MB)
 - resnet18_224x224_nv12_optimized_float_model.onnx (42.64 MB)
 - resnet18_224x224_nv12_original_float_model.onnx (42.64 MB)
 - resnet18_224x224_nv12_quant_info.json (0.02 MB)
 - resnet18_224x224_nv12_quantized_model.onnx (10.79 MB)
copied from: /home/ubuntu/horizon_x5_open_explorer_v1.2.8-py310_20240926/samples/ai_toolchain/horizon_model_convert_sample/03_classification/10_model_convert/mapper/model_output/resnet18_224x224_nv12.bin
copied to  : /home/ubuntu/model_latest.bin
copied to  : /home/ubuntu/resnet18_224x224_nv12_20260425_153317.bin


## 9. 备注

- 当前 `10_model_convert/mapper/postprocess.py` 还是分类样例，不适合直接解释这个 XY 回归模型的输出。
- 这个 notebook 已经保证导出的 ONNX/bin 输出是**原始坐标尺度**，不是标准化值。
- 如果后面你要做“板端推理结果可视化”，建议再单独补一个针对 `x,y` 的后处理脚本。
